# ICCIT 2026 — Leakage-Aware Benchmark

Full pipeline: data load → encoding → honest (group-aware) CV → leaky CV → TabNet → FT-Transformer → combined table → temporal split → figures.

**Deep models (TabNet, FT-Transformer) want a GPU.** Each is 5 folds × 100 epochs.

**Paper:** accepted for oral presentation at ICCIT 2026. This notebook reproduces the original benchmark (paper Table II). The camera-ready analyses (signature overlap, sensitivity/specificity, repeated temporal runs, expanding window, statistical tests) and the final Fig. 3 with error bars are in `03_camera_ready_analysis.ipynb`.

In [ ]:
# Setup: clone the repo so relative paths work (skip if already running inside it, e.g. locally)
import os
if not os.path.exists('data/thesis_dataset.csv'):
    !git clone -q https://github.com/Al-AminKhan043/leakage-aware-tabular-benchmark.git
    %cd leakage-aware-tabular-benchmark


In [ ]:
# Safety net: install anything missing (Colab already has most of these; matters for local runs)
import importlib.util
_missing = [p for p in ["xgboost", "lightgbm"] if importlib.util.find_spec(p) is None]
if _missing:
    %pip install -q {" ".join(_missing)}


In [ ]:
import pandas as pd
df= pd.read_csv('data/thesis_clean.csv')
assert "signature" in df.columns and df["signature"].nunique() == 235
print(df.shape)

## Honest split: group-aware holdout by signature (sanity check: zero overlap)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups=df['signature']
y=df['Quality']


gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, y, groups))

train_df = df.iloc[train_idx]
test_df  = df.iloc[test_idx]

overlap = set(train_df["signature"]) & set(test_df["signature"])
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Signature overlap (must be 0):", len(overlap))


## Encode categorical features

In [ ]:
from sklearn.preprocessing import LabelEncoder

drop_cols = ["Date", "Quality", "signature"]
feature_cols = [c for c in df.columns if c not in drop_cols]

df_enc = df.copy()
for col in feature_cols:
    if not pd.api.types.is_numeric_dtype(df_enc[col]):
        df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

df_enc["Quality"] = LabelEncoder().fit_transform(df_enc["Quality"].astype(str))

X_train = df_enc.iloc[train_idx][feature_cols]
X_test  = df_enc.iloc[test_idx][feature_cols]
y_train = df_enc.iloc[train_idx]["Quality"]
y_test  = df_enc.iloc[test_idx]["Quality"]

print("Non-numeric columns left:", [c for c in feature_cols if not pd.api.types.is_numeric_dtype(X_train[c])])
print("X_train dtypes:\n", X_train.dtypes)

## Quick single-model check: Random Forest under the honest split

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

print("RF accuracy:", round(accuracy_score(y_test, pred), 4))
print("RF macro-F1:", round(f1_score(y_test, pred, average="macro"), 4))

## SHAP feature attribution (honest-protocol Random Forest)

In [ ]:
# ============================================================
# SHAP feature attribution on the honest-protocol RandomForest.
# Explains WHICH features the honest model relies on -> shows it
# learns genuine chemical/procurement signal, not identity leakage.
# ============================================================
try:
    import shap
except ImportError:
    !pip install shap -q
    import shap

import os
os.makedirs("results/figures", exist_ok=True)
import matplotlib.pyplot as plt
import numpy as np

# TreeExplainer is exact + fast for RandomForest (no approximation).
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# RF binary classifier returns a list [class0_vals, class1_vals];
# we explain the positive ("Good"=1) class. Newer shap versions may
# return a single 3D array instead — handle both.
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values[..., 1] if shap_values.ndim == 3 else shap_values

plt.figure()
shap.summary_plot(
    sv, X_test,
    feature_names=feature_cols,
    show=False,
    plot_size=(7.0, 4.5),
)
plt.tight_layout()
plt.savefig("results/figures/shap_summary.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved -> results/figures/shap_summary.png")

## Full honest benchmark: 5-fold group-aware CV, 7 classical/MLP models

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# encoded features + target + group key
X = df_enc[feature_cols]
y = df_enc["Quality"]
groups = df["signature"]

# scale only for the models that need it (linear / distance / neural)
models = {
    "LogReg":       make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "SVM":          make_pipeline(StandardScaler(), SVC(random_state=42)),
    "AdaBoost":     AdaBoostClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "XGBoost":      XGBClassifier(random_state=42, n_jobs=-1, eval_metric="logloss"),
    "LightGBM":     LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    "MLP":          make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, model in models.items():
    accs, f1s = [], []
    for tr, te in cv.split(X, y, groups):
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict(X.iloc[te])
        accs.append(accuracy_score(y.iloc[te], p))
        f1s.append(f1_score(y.iloc[te], p, average="macro"))
    rows.append({
        "Model": name,
        "Accuracy_mean": np.mean(accs),
        "Accuracy_std":  np.std(accs),
        "MacroF1_mean":  np.mean(f1s),
        "MacroF1_std":   np.std(f1s),
    })
    print(f"{name:13s} acc {np.mean(accs):.4f} ± {np.std(accs):.4f}   macroF1 {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

results_df = pd.DataFrame(rows).sort_values("MacroF1_mean", ascending=False).reset_index(drop=True)
print("\n=== HONEST BENCHMARK (group-aware 5-fold CV) ===")
print(results_df.round(4).to_string(index=False))

## Save the honest benchmark table

In [ ]:
import os
os.makedirs("results/tables", exist_ok=True)
results_df.round(4).to_csv("results/tables/honest_benchmark.csv", index=False)
print("saved ->", "results/tables/honest_benchmark.csv")

## Leaky benchmark: standard (non-grouped) 5-fold CV, same models

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv_leaky = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows_leaky = []
for name, model in models.items():
    accs, f1s = [], []
    for tr, te in cv_leaky.split(X, y):
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict(X.iloc[te])
        accs.append(accuracy_score(y.iloc[te], p))
        f1s.append(f1_score(y.iloc[te], p, average="macro"))
    rows_leaky.append({"Model": name, "Leaky_Acc": np.mean(accs), "Leaky_F1": np.mean(f1s)})
    print(f"{name:13s} acc {np.mean(accs):.4f}   macroF1 {np.mean(f1s):.4f}")

leaky_df = pd.DataFrame(rows_leaky)

# merge into one side-by-side table
honest_df = results_df.rename(columns={"Accuracy_mean":"Honest_Acc","MacroF1_mean":"Honest_F1"})[["Model","Honest_Acc","Honest_F1"]]
compare = honest_df.merge(leaky_df, on="Model")
compare["Acc_Drop"] = (compare["Leaky_Acc"] - compare["Honest_Acc"]).round(4)
print("\n=== LEAKY vs HONEST ===")
print(compare.round(4).to_string(index=False))

## Save the leaky-vs-honest comparison table

In [ ]:
import os
os.makedirs("results/tables", exist_ok=True)
results_df.round(4).to_csv("results/tables/honest_benchmark.csv", index=False)
compare.round(4).to_csv("results/tables/leaky_vs_honest.csv", index=False)
print("saved both tables ->", os.listdir("results/tables"))

## TabNet setup

**GPU recommended** — in Colab: `Runtime > Change runtime type > GPU`.

In [ ]:
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
except ImportError:
    !pip install pytorch-tabnet -q
    from pytorch_tabnet.tab_model import TabNetClassifier

import torch
print("CUDA available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# arrays (TabNet wants numpy, not pandas)
X_np = df_enc[feature_cols].values.astype("float32")
y_np = df_enc["Quality"].values.astype("int64")
groups_np = df["signature"].values

# tell TabNet which columns are categorical so it uses embeddings (proper handling)
cat_cols  = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]
cat_idxs  = [feature_cols.index(c) for c in cat_cols]
cat_dims  = [int(df_enc[c].nunique()) for c in cat_cols]
print("Categorical cols:", cat_cols)
print("cat_idxs:", cat_idxs)
print("cat_dims:", cat_dims)

## TabNet: honest vs. leaky (5-fold each)

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

def run_tabnet(splitter, use_groups):
    accs, f1s = [], []
    split_iter = splitter.split(X_np, y_np, groups_np) if use_groups else splitter.split(X_np, y_np)
    for tr, te in split_iter:
        clf = TabNetClassifier(
            cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
            seed=42, verbose=0, device_name="auto"
        )
        clf.fit(X_np[tr], y_np[tr], max_epochs=100, patience=0,
                batch_size=256, virtual_batch_size=64)
        p = clf.predict(X_np[te])
        accs.append(accuracy_score(y_np[te], p))
        f1s.append(f1_score(y_np[te], p, average="macro"))
    return np.mean(accs), np.std(accs), np.mean(f1s), np.std(f1s)

h = run_tabnet(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42), use_groups=True)
print(f"TabNet HONEST  acc {h[0]:.4f} ± {h[1]:.4f}   macroF1 {h[2]:.4f} ± {h[3]:.4f}")

l = run_tabnet(StratifiedKFold(n_splits=5, shuffle=True, random_state=42), use_groups=False)
print(f"TabNet LEAKY   acc {l[0]:.4f} ± {l[1]:.4f}   macroF1 {l[2]:.4f} ± {l[3]:.4f}")

print(f"\nTabNet Acc_Drop (leaky - honest): {l[0]-h[0]:.4f}")

## FT-Transformer: honest vs. leaky (5-fold each)

In [ ]:
# ============================================================
# FT-Transformer  (Gorishniy et al. 2021 -> \\cite{gorishniy2021revisiting})
# Mirrors the TabNet cells. Reuses X_np, y_np, groups_np, cat_idxs,
# cat_dims, feature_cols from the TabNet setup cell.
# ============================================================
try:
    from rtdl_revisiting_models import FTTransformer
except ImportError:
    !pip install rtdl_revisiting_models -q
    from rtdl_revisiting_models import FTTransformer

import torch, torch.nn.functional as F, numpy as np
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

device   = "cuda" if torch.cuda.is_available() else "cpu"
n_classes = int(len(np.unique(y_np)))
cont_idxs = [i for i in range(len(feature_cols)) if i not in cat_idxs]   # continuous = the rest

def _train_predict_ftt(Xtr, ytr, Xte, n_epochs=100, lr=1e-3, batch_size=256):
    # FT-Transformer takes continuous + categorical as SEPARATE tensors.
    # Scale continuous on train stats only (FT-T needs standardized inputs).
    scaler   = StandardScaler().fit(Xtr[:, cont_idxs])
    Xtr_cont = torch.tensor(scaler.transform(Xtr[:, cont_idxs]), dtype=torch.float32, device=device)
    Xte_cont = torch.tensor(scaler.transform(Xte[:, cont_idxs]), dtype=torch.float32, device=device)
    Xtr_cat  = torch.tensor(Xtr[:, cat_idxs], dtype=torch.long, device=device) if cat_idxs else None
    Xte_cat  = torch.tensor(Xte[:, cat_idxs], dtype=torch.long, device=device) if cat_idxs else None
    ytr_t    = torch.tensor(ytr, dtype=torch.long, device=device)

    model = FTTransformer(
        n_cont_features=len(cont_idxs),
        cat_cardinalities=cat_dims,
        d_out=n_classes,
        **FTTransformer.get_default_kwargs(),
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)

    n = Xtr_cont.shape[0]
    model.train()
    for _ in range(n_epochs):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            b = perm[i:i+batch_size]
            opt.zero_grad()
            out = model(Xtr_cont[b], Xtr_cat[b] if Xtr_cat is not None else None)
            F.cross_entropy(out, ytr_t[b]).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return model(Xte_cont, Xte_cat).argmax(1).cpu().numpy()

def run_ftt(splitter, use_groups):
    accs, f1s = [], []
    it = splitter.split(X_np, y_np, groups_np) if use_groups else splitter.split(X_np, y_np)
    for tr, te in it:
        pred = _train_predict_ftt(X_np[tr], y_np[tr], X_np[te])
        accs.append(accuracy_score(y_np[te], pred))
        f1s.append(f1_score(y_np[te], pred, average="macro"))
    return np.mean(accs), np.std(accs), np.mean(f1s), np.std(f1s)

h_ftt = run_ftt(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42), use_groups=True)
l_ftt = run_ftt(StratifiedKFold(n_splits=5, shuffle=True, random_state=42),      use_groups=False)
print(f"FT-Transformer HONEST  acc {h_ftt[0]:.4f} +/- {h_ftt[1]:.4f}   macroF1 {h_ftt[2]:.4f}")
print(f"FT-Transformer LEAKY   acc {l_ftt[0]:.4f} +/- {l_ftt[1]:.4f}   macroF1 {l_ftt[2]:.4f}")
print(f"FT-Transformer Acc_Drop (leaky - honest): {l_ftt[0]-h_ftt[0]:.4f}")


## Combine all nine models into one final honest-vs-leaky table

In [ ]:
import pandas as pd, os

# Both deep models are logged from their REAL runs above
# (TabNet from the TabNet cell -> h,l ; FT-T from the cell above -> h_ftt,l_ftt).
tabnet_row = pd.DataFrame([{
    "Model": "TabNet",
    "Honest_Acc": round(h[0], 4),     "Honest_F1": round(h[2], 4),
    "Leaky_Acc":  round(l[0], 4),     "Leaky_F1":  round(l[2], 4),
    "Acc_Drop":   round(l[0]-h[0], 4),
}])
ftt_row = pd.DataFrame([{
    "Model": "FT-Transformer",
    "Honest_Acc": round(h_ftt[0], 4), "Honest_F1": round(h_ftt[2], 4),
    "Leaky_Acc":  round(l_ftt[0], 4), "Leaky_F1":  round(l_ftt[2], 4),
    "Acc_Drop":   round(l_ftt[0]-h_ftt[0], 4),
}])

final = pd.concat([compare, tabnet_row, ftt_row], ignore_index=True) \
          .sort_values("Honest_F1", ascending=False).reset_index(drop=True)

os.makedirs("results/tables", exist_ok=True)
final.round(4).to_csv("results/tables/final_benchmark.csv", index=False)
print(final.round(4).to_string(index=False))
print("\nsaved -> results/tables/final_benchmark.csv")


## Figure: leaky vs. honest accuracy, with majority-baseline and per-signature-oracle reference lines

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# majority baseline COMPUTED (fixes the old hardcoded 0.752 bug)
maj = df["Quality"].value_counts(normalize=True).max()                       # -> 0.636
# per-signature oracle ceiling = predict each signature's majority label
sig_major = df.groupby("signature")["Quality"].agg(lambda s: s.value_counts().index[0])
oracle = (df["Quality"].values == df["signature"].map(sig_major).values).mean()  # -> 0.942

models_order = final["Model"].tolist()
honest = final["Honest_Acc"].values
leaky  = final["Leaky_Acc"].values
x = np.arange(len(models_order)); w = 0.38

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(x - w/2, leaky,  w, label="Random split (leaky)", color="#d1495b")
ax.bar(x + w/2, honest, w, label="Group-aware (honest)", color="#2e7d8a")
ax.axhline(maj,    ls="--", lw=1.3, color="gray", label=f"Majority baseline ({maj*100:.1f}%)")
ax.axhline(oracle, ls=":",  lw=1.3, color="#444",  label=f"Per-signature oracle ceiling ({oracle*100:.1f}%)")
ax.set_xticks(x); ax.set_xticklabels(models_order, rotation=30, ha="right")
ax.set_ylabel("Accuracy"); ax.set_ylim(0.5, 1.0)
ax.set_title("Leaky vs. honest evaluation: accuracy collapse is concentrated in high-capacity models")
ax.legend(loc="upper right", framealpha=0.95)
plt.tight_layout()
import os; os.makedirs("results/figures", exist_ok=True)
plt.savefig("results/figures/leaky_vs_honest.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved -> results/figures/leaky_vs_honest.png")


## Figure: signature duplication diagnosis (sample-size collapse + repetition histogram)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# signature frequency = how many rows collapse into each unique signature
sig_counts = df["signature"].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ---- LEFT: the collapse, 9310 rows -> 235 unique signatures ----
bars = ax1.bar(["Total rows", "Unique\nsignatures"], [len(df), df["signature"].nunique()],
               color=["#d1495b", "#2e7d8a"], width=0.6)
ax1.set_ylabel("Count")
ax1.set_title("(a) Effective sample size after removing duplicates")
for b, v in zip(bars, [len(df), df["signature"].nunique()]):
    ax1.text(b.get_x() + b.get_width()/2, v + 120, f"{v:,}", ha="center", fontweight="bold")
# annotate the duplication rate
dup_rate = 100 * (1 - df["signature"].nunique() / len(df))
ax1.text(0.5, len(df)*0.55, f"{dup_rate:.1f}%\nduplication", ha="center",
         color="#d1495b", fontweight="bold", fontsize=12)
ax1.set_ylim(0, len(df)*1.1)

# ---- RIGHT: distribution of how many rows each signature repeats into ----
ax2.hist(sig_counts.values, bins=40, color="#2e7d8a", edgecolor="white")
ax2.set_xlabel("Rows per unique signature")
ax2.set_ylabel("Number of signatures")
ax2.set_title("(b) Signature repetition distribution")
ax2.axvline(sig_counts.mean(), ls="--", color="#d1495b",
            label=f"mean = {sig_counts.mean():.1f} rows/signature")
ax2.legend()

plt.tight_layout()
import os; os.makedirs("results/figures", exist_ok=True)
plt.savefig("results/figures/leakage_diagnosis.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved -> results/figures/leakage_diagnosis.png")

## Temporal protocol: train on earliest 80%, test on latest 20% (classical + MLP models)

In [ ]:
# ============================================================
# TEMPORAL protocol (train earliest 80% -> test latest 20%) for the
# classical + MLP models. Answers "is the leakage only duplication, or time?"
# ============================================================
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd, os

dft = df.copy()
dft["dt"] = pd.to_datetime(dft["Date"], format="%m/%d/%Y", errors="coerce")
order_idx = dft.sort_values("dt").index

X_sorted   = df_enc.loc[order_idx, feature_cols].reset_index(drop=True)
y_sorted   = df_enc.loc[order_idx, "Quality"].reset_index(drop=True)
sig_sorted = df.loc[order_idx, "signature"].reset_index(drop=True)

cut = int(len(X_sorted) * 0.8)
Xtr, Xte = X_sorted.iloc[:cut], X_sorted.iloc[cut:]
ytr, yte = y_sorted.iloc[:cut], y_sorted.iloc[cut:]

# diagnostic: how many test signatures already appeared before the cut?
sig_tr, sig_te = set(sig_sorted.iloc[:cut]), set(sig_sorted.iloc[cut:])
leak = len(sig_tr & sig_te)
print(f"Unique test signatures: {len(sig_te)} | seen before cut: {leak} "
      f"({leak/len(sig_te)*100:.0f}%)   <-- vs ~97% under a random split")
print(f"Test-set majority baseline: {max(yte.mean(), 1-yte.mean()):.4f}\n")

rows = []
for name, model in models.items():
    model.fit(Xtr, ytr)
    p = model.predict(Xte)
    rows.append({"Model": name,
                 "Temporal_Acc": round(accuracy_score(yte, p), 4),
                 "Temporal_F1":  round(f1_score(yte, p, average="macro"), 4)})
temporal_df = pd.DataFrame(rows)
print(temporal_df.sort_values("Temporal_F1", ascending=False).to_string(index=False))


## FT-Transformer: temporal protocol

In [ ]:
# FT-Transformer temporal row (single past/future split), appended to temporal_df.
import pandas as pd, os
from sklearn.metrics import accuracy_score, f1_score

dft = df.copy()
dft["dt"] = pd.to_datetime(dft["Date"], format="%m/%d/%Y", errors="coerce")
order_idx = dft.sort_values("dt").index.to_numpy()
cut = int(len(order_idx) * 0.8)
tr_idx, te_idx = order_idx[:cut], order_idx[cut:]

pred = _train_predict_ftt(X_np[tr_idx], y_np[tr_idx], X_np[te_idx])
ftt_temporal = {
    "Model": "FT-Transformer",
    "Temporal_Acc": round(accuracy_score(y_np[te_idx], pred), 4),
    "Temporal_F1":  round(f1_score(y_np[te_idx], pred, average="macro"), 4),
}
print(ftt_temporal)
temporal_df = pd.concat([temporal_df, pd.DataFrame([ftt_temporal])], ignore_index=True)
os.makedirs("results/tables", exist_ok=True)
temporal_df.to_csv("results/tables/temporal_benchmark.csv", index=False)

print("\nsaved -> results/tables/temporal_benchmark.csv")
print(temporal_df.sort_values("Temporal_F1", ascending=False).to_string(index=False))


## TabNet: temporal protocol

In [ ]:
# TabNet TEMPORAL protocol (train earliest 80% -> test latest 20%)
dft = df.copy()
dft["dt"] = pd.to_datetime(dft["Date"], format="%m/%d/%Y", errors="coerce")
order_idx = dft.sort_values("dt").index.to_numpy()
cut = int(len(order_idx) * 0.8)
tr_idx, te_idx = order_idx[:cut], order_idx[cut:]

clf = TabNetClassifier(
    cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
    seed=42, verbose=0, device_name="auto"
)
clf.fit(X_np[tr_idx], y_np[tr_idx], max_epochs=100, patience=0,
        batch_size=256, virtual_batch_size=64)
pred = clf.predict(X_np[te_idx])

tabnet_temporal = {
    "Model": "TabNet",
    "Temporal_Acc": round(accuracy_score(y_np[te_idx], pred), 4),
    "Temporal_F1":  round(f1_score(y_np[te_idx], pred, average="macro"), 4),
}
print(tabnet_temporal)

temporal_df = pd.concat([temporal_df, pd.DataFrame([tabnet_temporal])], ignore_index=True)
temporal_df.to_csv("results/tables/temporal_benchmark.csv", index=False)
print(temporal_df.sort_values("Temporal_F1", ascending=False).to_string(index=False))

## Figure: all three protocols (leaky / honest / temporal) side by side — single temporal run (final paper figure: see notebook 03)

In [ ]:
# ============================================================
# THREE-PROTOCOL figure (single temporal run), leaky vs honest vs temporal.
# The paper's final Fig. 3 (repeated temporal, error bars) is made in 03_camera_ready_analysis.ipynb.
# No internal title (the LaTeX caption carries the description).
# ============================================================
import matplotlib.pyplot as plt
import numpy as np, os

plot_df = final.merge(temporal_df, on="Model", how="left")
order   = plot_df.sort_values("Honest_Acc", ascending=False)["Model"].tolist()
plot_df = plot_df.set_index("Model").loc[order].reset_index()

maj = df["Quality"].value_counts(normalize=True).max()
sig_major = df.groupby("signature")["Quality"].agg(lambda s: s.value_counts().index[0])
oracle = (df["Quality"].values == df["signature"].map(sig_major).values).mean()

x = np.arange(len(plot_df)); w = 0.26
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.bar(x - w, plot_df["Leaky_Acc"],    w, label="Random split (leaky)",      color="#d1495b")
ax.bar(x,     plot_df["Honest_Acc"],   w, label="Group-aware (honest)",      color="#2e7d8a")
ax.bar(x + w, plot_df["Temporal_Acc"], w, label="Temporal (future-holdout)", color="#e3a13c")
ax.axhline(maj,    ls="--", lw=1.3, color="gray", label=f"Majority baseline ({maj*100:.1f}%)")
ax.axhline(oracle, ls=":",  lw=1.3, color="#444",  label=f"Per-signature oracle ceiling ({oracle*100:.1f}%)")
ax.set_xticks(x); ax.set_xticklabels(plot_df["Model"], rotation=30, ha="right")
ax.set_ylabel("Accuracy"); ax.set_ylim(0.5, 1.0)
ax.legend(loc="upper right", framealpha=0.95)
plt.tight_layout()
os.makedirs("results/figures", exist_ok=True)
plt.savefig("results/figures/three_protocol_single_run.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved -> results/figures/three_protocol_single_run.png")
